# Car Price Prediction


#### Project Workflow
1. Understand the Dataset
- Review all columns and their meanings (you’ve already done this — great start!)
- Identify which variables are:
- Independent (features)
- Dependent (target)

2. Clean and Prepare the Data
- Check for missing values or anomalies (e.g., nulls sales)
- Create new features if needed:

3. Explore the Data (EDA)
Use visualizations to uncover patterns:
- 📉 Boxplots to see sales distribution by weather or promotion
- 📌 Correlation heatmap to see which features influence sales most

4. Model Sales Drivers
- Use regression models (e.g., Linear Regression, Random Forest, XGBoost) to predict daily_sales
- Evaluate feature importance: which variables drive sales the most?
- Try time series models (e.g., ARIMA, Prophet) if you're forecasting future sales

7. Present Your Work
- Create a dashboard (Excel, Power BI, or Tableau)
- Summarize key insights in a slide deck or report
- Include visuals, trends, and actionable takeaways


In [ ]:
# Importing Libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [ ]:
#Importing Dataset
df = pd.read_csv('car_sales_data.csv')

raw_df = df.copy()

## Data Cleaning

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
# Handling Duplicates
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
# Handling Missing Values
df.isnull().sum()

## EDA


✅ 1. Univariate Analysis (Quick Checks)
✔ Histograms / KDE plots for:
- Price
- Mileage
- Year of manufacture
- Engine size


✅ 2. Outlier Detection (Outliers can ruin models if not handled.)
Use: Boxplots, IQR, Scatterplots of Price vs key features
Look specifically for:
- Extremely old cars
- Very high mileage cars
- Price values too low or too high compared to the rest


✅ 3. Relationship Analysis (Minimal Pairwise Checks)
Only 3 scatterplots are needed:
- Price vs Mileage (declining trend?)
- Price vs Age/Year (strong expected relationship)
- Price vs Engine size (positive relationship)
Purpose:
Helps you see whether linear or non-linear models might work better, and where transformations may help.


✅ 4. Categorical Variable Insights
✔ View counts of: Manufacturer, Model, Fuel type
`df['Manufacturer'].value_counts()`
`df['Model'].value_counts()`
`df['Fuel type'].value_counts()`

Why this matters:
- Helps detect rare categories
- Helps decide which encoding to use
- Model will likely need target/frequency encoding
- Manufacturer/Fuel type → one-hot is fine


✅ 5. Multicollinearity Check (Optional but Recommended)

Mainly to avoid redundant variables:

- Year vs Car age (if you eventually create it)
- Engine size vs Engine category
- Mileage vs Mileage per year (after engineering)

For now, just check correlation among numerical columns


`NB: Do No7 after Completing Data Preprocessing and Feature Engineering`

In [ ]:
# Histogram + KDE
plt.subplot(1, 2, 1)
sns.histplot(df[df['Engine size', 'Year of manufacture', 'Mileage', 'Price']], kde=True)
plt.title('Distribution of Numerical Features')


In [ ]:
# plot a histogram for each numerical attribute
df.hist(bins=50, figsize=(20,15))
plt.show()



In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
# Summarizing numerical columns
print(df[['Engine size','Year of manufacture', 'Mileage', 'Price']].describe())


# Visualizing boxplots for numerical columns
for col in ['Engine size','Year of manufacture', 'Mileage', 'Price']:
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

### Looking for Correlations

In [ ]:
df.dtypes

In [ ]:
corr_matrix = df[['Engine size', 'Year of manufacture', 'Mileage', 'Price']].corr()

In [ ]:
# Visualizing the correlation matrix using a heatmap

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Features')
plt.show()

From the correlation matrix above;
- Year of manufacture has a strong direct proportionality to Prie
- Engine size is moderately directly proportional to price
- Mileage is Strongly Indirectly Proportional to the Price

In [ ]:
df.head(10)

### Univariate Analysis (Quick Checks)
✔ Histograms / KDE plots for:
- Price
- Mileage
- Year of manufacture
- Engine size

In [ ]:
# Univariate Analysis for price

plt.figure(figsize=(14,5))

# Histogram + KDE
plt.subplot(1, 2, 1)
sns.histplot(df['Price'], kde=True)
plt.title('Distribution of Car Price')

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=df['Price'])
plt.title('Car Price Outliers')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.random.randn(1000) # Example data
plt.hist(data, bins=30, edgecolor='black')
plt.title('Histogram of Data')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

data = np.random.randn(1000) # Example data
sns.kdeplot(data, fill=True)
plt.title('KDE Plot of Data')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

# Data Preprocessing

✅ 1. Essential Feature Engineering for Car Price Prediction

- Car Age (instead of using Year directly)
`df['Car_Age'] = current_year - df['Year of manufacture']`

- Mileage per Year (Usage Intensity). i.e A car with 150,000 km over 15 years is not the same as 150,000 km over 5 years.
`df['Mileage_per_Year'] = df['Mileage'] / df['Car_Age'].replace(0, 1)`

- Engine Size Category (optional binning)
You can create bins like:
Small: <1.4
Medium: 1.4–2.0
Large: >2.0
Or just standard binning:
`df['Engine_Category'] = pd.cut(df['Engine size'], bins=[0, 1.4, 2.0, 4.0], labels=['Small','Medium','Large'])`

- Log-transform the target (Price): Car prices are usually right-skewed. Applying log transformation stabilizes variance and reduces outlier impact
`df['Log_Price'] = np.log(df['Price'])`


✅ 2. Encoding Categorical Variables Properly
Thefore the ffg encoding is prefarable;
Manufacturer → One-hot encoding is fine.
Fuel Type → One-hot encoding works.
Model → This is tricky.


✅ 3. Interaction features (often useful)
Useful examples:
Engine size × Manufacturer
Engine size × Fuel type
Car age × Mileage

✅ 4. Remove or treat extreme outliers.
Before fitting the model:
-Remove cars priced at absurdly low or high values
- Remove mileage above the 99th percentile
- Remove cars older than 30 years if they skew the model

eg: Your dataset has a 1988 Toyota — that’s essentially a classic/vintage category and will distort the model.